In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
FOLDER = os.environ.get("UNSW_FOLDER", r"C:\Users\Lenovo\Documents\pbl\dataset one")
 
# Time split on the 18 Feb session (UTC, same clock as Stime)
SESSION_START = os.environ.get("SESSION_START", "2015-02-18 00:00")
TRAIN_END     = os.environ.get("TRAIN_END",     "2015-02-18 05:00")
VAL_END       = os.environ.get("VAL_END",       "2015-02-18 07:00")
 
MIN_CAT_COUNT = 1000        # categories rarer than this in TRAIN become "other"
SKEW_LIMIT    = 5

In [3]:
CATEGORICAL = ["proto", "state", "service"]
NOT_FEATURES = ["srcip", "dstip", "sport", "dsport", "Stime", "Ltime", "attack_cat", "Label"]
META_COLS = ["Stime", "srcip", "dstip", "sport", "dsport", "service", "attack_cat", "Label"]

In [4]:
def load_clean():
    for name in ["unsw_clean.parquet", "unsw_clean.pkl"]:
        path = os.path.join(FOLDER, name)
        if os.path.exists(path):
            print("loading", path)
            return pd.read_parquet(path) if name.endswith(".parquet") else pd.read_pickle(path)
    raise FileNotFoundError("run step1_load_clean.py first (no unsw_clean file found)")

In [5]:
def time_split(df):
    t0, t1, t2 = map(pd.Timestamp, [SESSION_START, TRAIN_END, VAL_END])
    in_session = df["Stime"] >= t0
    parts = {
        "train": df[in_session & (df["Stime"] < t1)],
        "val":   df[(df["Stime"] >= t1) & (df["Stime"] < t2)],
        "stream": df[df["Stime"] >= t2],
    }
    print(f"rows not used (before {SESSION_START}): {(~in_session).sum():,}")
    for k, p in parts.items():
        print(f"{k:7s} rows={len(p):>9,}  attacks={int(p['Label'].sum()):>8,} "
              f"({p['Label'].mean():.1%})  {p['Stime'].min()} -> {p['Stime'].max()}")
    assert all(len(p) > 0 for p in parts.values()), "a split is empty: check the cut times"
 
    print("\nattack_cat counts per split:")
    table = pd.DataFrame({k: p["attack_cat"].value_counts() for k, p in parts.items()}).fillna(0).astype(int)
    print(table)
    missing = [c for c in table.index if c != "Normal" and table.loc[c, "train"] == 0]
    if missing:
        print("WARNING: attack types with ZERO rows in train:", missing)
    return parts

In [6]:
def fit_categories(train):
    cats = {}
    for col in CATEGORICAL:
        vc = train[col].astype(str).value_counts()
        cats[col] = {"keep": vc[vc >= MIN_CAT_COUNT].index.tolist(),
                     "seen": set(vc.index)}
        print(f"{col}: keeping {len(cats[col]['keep'])} of {len(vc)} categories")
    return cats

In [7]:
def encode_categoricals(part, cats):
    pieces = []
    for col in CATEGORICAL:
        s = part[col].astype(str)
        unseen = ~s.isin(cats[col]["seen"])
        if unseen.any():
            print(f"   {col}: {unseen.sum():,} rows have a category never seen in train")
        s = s.where(s.isin(cats[col]["keep"]), "other")
        levels = cats[col]["keep"] + ["other"]
        dummies = pd.get_dummies(pd.Categorical(s, categories=levels), prefix=col, dtype="int8")
        dummies.index = part.index
        pieces.append(dummies)
    return pd.concat(pieces, axis=1)

In [8]:
def numeric_frame(part):
    return part.drop(columns=NOT_FEATURES + CATEGORICAL).astype("float64")
 
 
def fit_numeric(train_num):
    log_cols = [c for c in train_num.columns
                if train_num[c].min() >= 0 and train_num[c].nunique() > 2
                and train_num[c].skew() > SKEW_LIMIT]
    print(f"log1p applied to {len(log_cols)} skewed columns:", log_cols)
    t = train_num.copy()
    t[log_cols] = np.log1p(t[log_cols])
    lo, hi = t.min(), t.max()
    rng = (hi - lo).replace(0, 1.0)
    const = (hi - lo == 0)
    if const.any():
        print("constant-in-train columns (scaled to 0):", const[const].index.tolist())
    return {"log_cols": log_cols, "min": lo, "range": rng}

In [9]:
def apply_numeric(num, p):
    num = num.copy()
    num[p["log_cols"]] = np.log1p(np.maximum(num[p["log_cols"]], 0))
    return (num - p["min"]) / p["range"]          # deliberately NOT clipped
 
 
def duplicate_report(raw):
    h = {k: pd.util.hash_pandas_object(numeric_frame(p).join(p[CATEGORICAL].astype(str)), index=False)
         for k, p in raw.items()}
    for k in ["train", "val", "stream"]:
        print(f"{k:7s} duplicate rows inside split: {h[k].duplicated().sum():>8,} "
              f"({h[k].duplicated().mean():.1%})")
    tr = set(h["train"].values)
    for k in ["val", "stream"]:
        hit = h[k].isin(tr).sum()
        print(f"{k:7s} rows identical to a train row: {hit:>8,} ({hit / len(h[k]):.1%})")

In [10]:
def main():
    df = load_clean()
    raw = time_split(df)
 
    print("\n-- duplicates (report only, nothing is dropped) --")
    duplicate_report(raw)
 
    print("\n-- fitting preprocessing on TRAIN only --")
    cats = fit_categories(raw["train"])
    num_params = fit_numeric(numeric_frame(raw["train"]))
 
    out = {"meta": {}, "X": {}, "y": {}}
    feature_names = None
    print("\n-- applying to each split --")
    for k, part in raw.items():
        print(k)
        X = pd.concat([apply_numeric(numeric_frame(part), num_params),
                       encode_categoricals(part, cats)], axis=1)
        if feature_names is None:
            feature_names = X.columns.tolist()
        assert X.columns.tolist() == feature_names
        assert np.isfinite(X.values).all(), f"non-finite values in {k}"
        out["X"][k] = X.values.astype("float32")
        out["y"][k] = part["Label"].values.astype("int8")
        out["meta"][k] = part[META_COLS].reset_index(drop=True)
 
    out["feature_names"] = feature_names
    out["params"] = {"categories": cats, "numeric": num_params,
                     "split": {"session_start": SESSION_START, "train_end": TRAIN_END, "val_end": VAL_END}}
 
    print(f"\nfinal feature count: {len(feature_names)}")
    for k in out["X"]:
        print(k, out["X"][k].shape, "min/max of scaled values:",
              round(float(out["X"][k].min()), 3), round(float(out["X"][k].max()), 3))
    path = os.path.join(FOLDER, "step2_data.pkl")
    pd.to_pickle(out, path)
    print("saved", path)
    return out

In [11]:
if __name__ == "__main__":
    out = main()

loading C:\Users\Lenovo\Documents\pbl\dataset one\unsw_clean.pkl
rows not used (before 2015-02-18 00:00): 1,087,203
train   rows=  491,162  attacks=  93,563 (19.0%)  2015-02-18 00:23:28 -> 2015-02-18 04:59:59
val     rows=  252,433  attacks=  60,609 (24.0%)  2015-02-18 05:00:00 -> 2015-02-18 06:59:59
stream  rows=  709,249  attacks= 144,896 (20.4%)  2015-02-18 07:00:00 -> 2015-02-18 12:21:08

attack_cat counts per split:
                 train     val  stream
attack_cat                            
Analysis           694     439    1018
Backdoor           437     345    1013
DoS               5154    2317    7715
Exploits         13963    7115   18038
Fuzzers           7037    3261    8897
Generic          61485   44771  101703
Normal          397599  191824  564353
Reconnaissance    4273    2109    5846
Shellcode          466     227     595
Worms               54      25      71

-- duplicates (report only, nothing is dropped) --
train   duplicate rows inside split:  136,211 (27.7%)
v